# EVASPA Evapotranspiration processing

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os
from pprint import pprint

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import ListedColormap

In [ ]:
from sweat.api import filter_data_for_evaspa
from sweat.common import flux, io
from sweat.evaspa import ef, merging, seb
from sweat.evaspa.config import EVASPAParamsConfig

### Common

In [ ]:
def plot_rgb(data: xr.Dataset, bands: list[str] | None = None):
    if bands is None:
        bands = ["red", "green", "blue"]
    rgb = xr.DataArray(
        np.dstack(
            (data[bands[0]].data, data[bands[1]].data, data[bands[2]].data)
        ),
        data.coords.assign(band=["r", "g", "b"]),
        dict(data.sizes, band=3),
    )
    rgb.plot.imshow(  # type: ignore
        rgb="band",
        vmin=rgb.quantile(0.01),
        vmax=rgb.quantile(0.99),
    )
    plt.title("RGB")


def plot_band(data: xr.Dataset, band: str, percentile=0):
    data[band].plot.imshow(  # type: ignore
        vmin=data[band].quantile(percentile),
        vmax=data[band].quantile(1 - percentile),
    )


def plot_mask(data: xr.Dataset, mask: str, invert=False):
    cmap = [(0.0, 0.0, 0.0, 0.0)]
    if len(np.unique(data[mask].data)) > 1:
        cmap = [(1.0, 0.0, 0.0, 1.0), (0.0, 0.0, 0.0, 0.0)]
    if invert:
        cmap = cmap[::-1]
    valid_cmap = ListedColormap(cmap)
    data[mask].plot(cmap=valid_cmap, add_colorbar=False)


def plot_models(
    data: xr.Dataset,
    models: xr.Dataset,
):
    """
    Plot EF models
    """
    # Set figure subplots
    nb = len(models.data_vars)
    row = int(np.ceil(nb / 2))
    col = 2
    fig = plt.figure(figsize=(4 * col, 3 * row), constrained_layout=True)

    i = 0
    j = 0
    # Loop over models
    for model_name in models.data_vars:
        if model_name in ["valid", "flags"]:
            continue
        model = ef.EFModel.create(models[model_name].attrs)
        var_name = model.var
        var = data[model.var].data
        lst = data["lst"].data
        ax = plt.subplot2grid((row, col), (i, j))
        # plot LST points
        ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)
        # edges coordinates
        var_max = np.round(np.ceil(np.nanmax(var) * 10) / 10, decimals=1)
        var_min = np.round(np.floor(np.nanmin(var) * 10) / 10, decimals=1)
        var_arr = np.linspace(var_min, var_max, num=100)
        tw_arr = model.twet(var_arr)
        td_arr = model.tdry(var_arr)
        ax.plot(
            var_arr,
            td_arr,
            color="tab:red",
            linewidth=3,
            label="dry edge",
            zorder=1,
        )
        ax.plot(
            var_arr,
            tw_arr,
            color="tab:blue",
            linewidth=3,
            label="wet edge",
            zorder=1,
        )
        # Labels
        ax.set_xlabel(var_name)
        ax.set_ylabel("Land Surface Temperature K")
        ax.set_title(f"Model {model_name}", fontsize=12)
        j += 1
        if j == 2:
            j = 0
            i += 1

    # title
    fig.suptitle("Evaporative fraction methods", fontsize=12)

    plt.show()


def plot_data(
    data: xr.Dataset,
    var_name: str,
):
    """
    Plot
    """
    # Figure dimensions
    _ = plt.figure(constrained_layout=True, figsize=(4, 3))

    ax = plt.gca()

    lst = data["lst"].data
    var = data[var_name].data
    # plot all points
    ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)

    # Labels
    ax.set_xlabel(var_name)
    ax.set_ylabel("Land Surface Temperature K")

    plt.show()


def plot_dataset(
    data: xr.Dataset,
):
    """
    Plot EF models
    """
    # Set figure subplots
    nb = len(data.data_vars)
    row = int(np.ceil(nb / 2))
    col = 2
    fig = plt.figure(figsize=(4 * col, 3 * row), constrained_layout=True)

    i = 0
    j = 0
    # Loop over variables
    for var in data.data_vars:
        ax = plt.subplot2grid((row, col), (i, j))
        if var == "valid":
            valid_cmap = ListedColormap(
                [(1.0, 0.0, 0.0, 1.0), (0.0, 1.0, 0.0, 1.0)]
            )
            data[var].plot(ax=ax, cmap=valid_cmap, add_colorbar=False)
        elif var == "flags":
            da = data[var]
            unique_combinations = np.unique(da.values)
            # Use a qualitative colormap with enough colors
            # (e.g., tab20 or a custom one)
            cmap = plt.get_cmap("tab20", len(unique_combinations))
            # Map each unique mask value to an integer index (for coloring)
            value_to_index = {v: i for i, v in enumerate(unique_combinations)}
            indexed_data = da.copy(
                data=np.vectorize(value_to_index.get)(da.values)
            )
            indexed_data.plot(ax=ax, cmap=cmap, add_colorbar=False)
            legend_patches = [
                mpatches.Patch(color=cmap(i), label=format(v, "08b"))
                for i, v in enumerate(unique_combinations)
            ]
            ax.legend(
                handles=legend_patches,
                title="Flags (bits)",
                bbox_to_anchor=(1, 1),
                loc="upper right",
                fontsize="small",
                fancybox=True,
                framealpha=0.5,
            )
        else:
            data[var].plot(ax=ax)
        j += 1
        if j == 2:
            j = 0
            i += 1
    # title
    fig.suptitle("Dataset", fontsize=12)

    plt.show()

### Input

Read data

In [ ]:
src_path = os.path.dirname(os.path.dirname(os.getcwd()))
data = io.read_data_from_file(
    os.path.join(src_path, "tests", "data", "modis_test_dem.tif")
)
# Add date to metadata
data.attrs["date"] = dt.datetime(2018, 5, 16, 10, 0, 0, tzinfo=dt.UTC)

In [ ]:
data

Generate cloud mask (for demonstration purpose)

In [ ]:
cloud_mask = xr.ones_like(data["lst"])
data["cloud"] = cloud_mask.where(
    (cloud_mask.x >= -16.2)
    & (cloud_mask.x <= -16)
    & (cloud_mask.y >= 14.7)
    & (cloud_mask.y <= 14.8),
    0,
)

Generate albedo invalid data (for demonstration purpose)

In [ ]:
data["albedo"] = data["albedo"].where(
    ~(
        (data["albedo"].x >= -16.85)
        & (data["albedo"].x <= -16.8)
        & (data["albedo"].y >= 14.4)
        & (data["albedo"].y <= 14.45)
    ),
    np.nan,
)

Plot data

In [ ]:
plot_band(data, band="lst")

In [ ]:
plot_band(data, band="cloud")

In [ ]:
plot_band(data, band="albedo")

In [ ]:
plot_data(data, "albedo")

Configuration

In [ ]:
params = {
    "filtering": {"cloud": {"op": "!=", "value": 1}},
    "ef": {
        "check": {"threshold": 0.02},
        "models": "default_evaspa",
        "options": {
            "filtering": {
                "albedo": {
                    "and": [
                        {"op": ">=", "value": 0.1},
                        {"op": "<=", "value": 0.3},
                    ]
                }
            },
            "selection": False,
            "merging": {
                "merging_method": "median",
                "uncertainty_method": "interquartile",
            },
        },
    },
    "seb": {"use_topo": False, "models": ["kustas"]},
}

## Run step by step 

Validate parameters config

In [ ]:
# Validate parameters config
params_config = EVASPAParamsConfig.model_validate(params)

In [ ]:
params_config

In [ ]:
pprint(params_config.model_dump(by_alias=True), sort_dicts=False)

Initalize

In [ ]:
models, options = ef.initialize(params_config.ef.model_dump(by_alias=True))

Filter

  - Identify valid pixels for edge computation
  - Identify invalid pixels

In [ ]:
data = filter_data_for_evaspa(
    data=data, models=models, config=params_config.filtering.model_dump()
)

Plot valid mask

In [ ]:
if (data["valid"].data == 0).sum() == 0:
    print("No invalid pixels")
else:
    plot_mask(data=data, mask="valid", invert=True)

Plot flags

In [ ]:
data["flags"].plot()

Check variability

In [ ]:
ef.check_variability(
    data["lst"], mask=data["valid"], **params_config.ef.check.model_dump()
)

Compute EF

In [ ]:
models

In [ ]:
ef_xr, ef_inst_xr = ef.run(
    models, data, **params_config.ef.options.model_dump(by_alias=True)
)

In [ ]:
ef_xr.data_vars

In [ ]:
ef_xr

In [ ]:
plot_models(data, ef_xr)

In [ ]:
plot_dataset(ef_xr)

Compute LE

In [ ]:
print(f"Constant Stefan Boltzmann = {flux.CST_SB}")

Compute net radiation

In [ ]:
rn_xr = seb.create_net_radiation(data, use_topo=True)

In [ ]:
rn_xr

In [ ]:
print(f"Max = {np.nanmax(rn_xr.rn)}")
print(f"Min = {np.nanmin(rn_xr.rn)}")

In [ ]:
plot_band(rn_xr, band="rn")

Compute G/Rn ratio

In [ ]:
ratio_xr = seb.create_ratio(data, models=params_config.seb.models)

In [ ]:
ratio_xr

In [ ]:
print(f"Max = {np.nanmax(ratio_xr.kustas)}")
print(f"Min = {np.nanmin(ratio_xr.kustas)}")

In [ ]:
plot_band(ratio_xr, band="kustas")

In [ ]:
le_xr = seb.create_le(ef_xr, rn_xr, ratio_xr)

In [ ]:
le_xr

In [ ]:
plot_dataset(le_xr)

Merge LE

In [ ]:
le_inst = merging.merge_to_dataset(
    le_xr,
    merging_method=merging.MergingMethod.MEDIAN,
    uncertainty_method=merging.UncertaintyMethod.INTERQUARTILE,
    name="le",
)

In [ ]:
le_inst

Visualize results

In [ ]:
plot_dataset(le_inst)